In [1]:
Q1 =  "I just discovered the course. Can I still join it?"
Q2 = "I just found out about the program. Can I still enroll?"

#These two are close - they mean the same thing.

Q3 = "How do I run Docker on Windows?"

#This one is far away from Q1 and Q2.

In [2]:
from sentence_transformers import SentenceTransformer
print("Loading model...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Encoding sentences...")
v1 = model.encode(Q1)

Loading model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding sentences...


In [4]:
v1.shape

(384,)

In [5]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [6]:
v1.dot(dv)

np.float32(0.38779125)

# Embedding Our Dataset

In [7]:
from ingest import load_faq_data

documents = load_faq_data()

In [8]:
# we have to get our text for the document

texts = []

for doc in documents:
    text = doc['question'] + " " + doc['answer']
    texts.append(text)


In [9]:
len(texts)

1350

In [10]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

# Vector Search

In [11]:
Q1 =  "I just discovered the course. Can I still join it?"
v1 = model.encode(Q1)

In [12]:
import numpy as np
X = np.array(vectors)

In [13]:
scores = X.dot(v1)

In [14]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(538), np.float32(0.8365045))

In [15]:
documents[idx]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [16]:
top5 = np.argsort(scores)[-5:] # devuelve una lista de los índices ordenados de menor a mayor
top5 = top5[::-1] # invertimos la lista para tener los índices de mayor a menor

In [17]:
scores[top5]

array([0.8365045 , 0.6903564 , 0.60425967, 0.59591275, 0.5927248 ],
      dtype=float32)

In [18]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.8365045
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

0.6903564
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'}

0.60425967
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 's

# Vector Search with minsearch

In [21]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [22]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [23]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [25]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [26]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

# RAG with Vector Search

In [27]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [28]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [29]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [30]:
query = "I just discovered the course. Can I still join it?"
assistant.rag(query)

'Yes, you can still join. If you want a certificate, you need to submit your project while submissions are still open.'

In [31]:
#Now we add the embedder
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [34]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [35]:
query = "I just discovered the course. Can I still join it?"
vector_assistant.rag(query)

'Yes, you can still join the course. If you want a certificate, make sure to submit your project while submissions are still open.'

# Vector Search with sqlitesearch

In [36]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [37]:
vs_index.fit(X, documents)

In [38]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [39]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [42]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [43]:
vs_index.close()

Ahora imaginamos que estamos en una terminal nueva. Vamos a hacer uso de que ya guardamos la base de datos vectorizada en el disco

In [44]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [45]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [46]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [47]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [48]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join even if the program has already begun. You can start learning and submitting homework while the submission form is open. If you want a certificate, make sure to submit your project before submissions close.'

# Vector Search with PGVector